# Étape 5 — Thèmes : TF-IDF/K-Means et BERTopic

Reprise des deux modèles de `../../bluesky_topics.ipynb`. K-Means est la référence légère. BERTopic utilise MiniLM multilingue → UMAP → HDBSCAN → c-TF-IDF, avec représentation MMR et bigrammes.

Prérequis : notebook 04 exécuté. BERTopic est activé explicitement avec `RUN_BERTOPIC` ; il nécessite `requirements-bertopic.txt` et peut télécharger le modèle lors de la première exécution.

In [ ]:
from pathlib import Path
import json
from IPython.display import display

cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "bluesky-news-mining", cwd / "textmining" / "bluesky-news-mining"]
ROOT = next((p.resolve() for p in candidates if (p / "notebooks" / "00_fonctions_base.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le sous-projet ou son dossier notebooks.")

def load_definitions(name, tag):
    path = ROOT / "notebooks" / name
    notebook = json.loads(path.read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] == "code" and tag in cell["metadata"].get("tags", []):
            exec(compile("".join(cell["source"]), str(path), "exec"), globals())

load_definitions("00_fonctions_base.ipynb", "database-definitions")
load_definitions("00_fonctions_analyse.ipynb", "analysis-definitions")

PROC = ROOT / "data" / "processed"
TAB = ROOT / "outputs" / "tables"
FIG = ROOT / "outputs" / "figures"
for directory in (PROC, TAB, FIG):
    directory.mkdir(parents=True, exist_ok=True)
print("Sous-projet :", ROOT.name)

In [ ]:
corpus_path = PROC / "corpus_analyse.csv"
if not corpus_path.exists():
    raise FileNotFoundError("Exécuter d'abord le notebook 04.")
df = pd.read_csv(corpus_path).fillna({"texte_clean": "", "text_emb": ""})
df["created_at"] = pd.to_datetime(df["created_at"], utc=True, format="mixed")
df["date"] = pd.to_datetime(df["date"])
print("Corpus chargé :", len(df), "posts")

## 1. Reference TF-IDF / K-Means

Traitement de la source : min_df=5, max_df=0.5, 14 groupes sans reduction automatique, vecteurs nuls inclus, silhouette euclidienne et stabilite ARI entre graines (n_init=3 pour les repetitions). Les corpus trop petits sont signales plutot que traites avec des parametres modifies.

In [ ]:
N_KMEANS = 14
MIN_DF = 5
if len(df) < 3:
    raise ValueError("Corpus insuffisant : élargir la collecte ou revoir les filtres du notebook 04.")
df, km_info, km_diagnostics = fit_kmeans_topics(df, n_clusters=N_KMEANS, min_df=MIN_DF)
display(km_info)
display(pd.Series(km_diagnostics, name="Valeur").to_frame())
df.to_csv(PROC / "post_topics.csv", index=False)
km_info.to_csv(TAB / "05_kmeans_themes.csv", index=False)
diagnostics = {"kmeans": km_diagnostics, "bertopic": {"status": "not_run"}}
# Supprimer les anciens résultats facultatifs pour éviter de les confondre avec ce run.
pd.DataFrame(columns=["topic","posts","authors","label","top_terms","examples"]).to_csv(
    TAB / "05_bertopic_themes.csv", index=False)
(PROC / "topic_diagnostics.json").write_text(json.dumps(diagnostics, indent=2), encoding="utf-8")

## 2. BERTopic : parametres source

UMAP conserve les 15 voisins, les 5 dimensions et son initialisation par defaut. HDBSCAN et MMR reprennent les reglages de la source. Le vectoriseur utilise STOP_ACC et min_df=5. REASSIGN_OUTLIERS=True active par defaut la reaffectation par embeddings au seuil 0.35. Les labels bruts et finaux sont separes. Si ces parametres eliminent tout le vocabulaire, un diagnostic explicite est exporte sans abaisser min_df automatiquement.

In [ ]:
RUN_BERTOPIC = True
TARGET_TOPICS = 30
MIN_CLUSTER_SIZE = 15
REASSIGN_OUTLIERS = True
OUTLIER_THRESHOLD = 0.35

topic_model = None
if RUN_BERTOPIC:
    from sentence_transformers import SentenceTransformer

    st_model = SentenceTransformer(EMB_MODEL)
    emb, cache_info = cached_embeddings(df, PROC / "embedding_cache", EMB_MODEL, model=st_model)
    try:
        df_bt, topic_info, bt_diagnostics, topic_model = fit_bertopic_topics(
            df, emb, st_model, target_topics=TARGET_TOPICS, min_cluster_size=MIN_CLUSTER_SIZE,
            reassign_outliers=REASSIGN_OUTLIERS, outlier_threshold=OUTLIER_THRESHOLD,
        )
    except SourceAnalysisNotApplicable as error:
        diagnostics["bertopic"] = {"status": "source_parameters_not_applicable", "reason": str(error)}
        print("BERTopic non calculable avec les parametres source :", error)
    else:
        df = df_bt
        mask = df["topic"] >= 0
        bt_diagnostics["ari_vs_kmeans"] = (
            float(adjusted_rand_score(df.loc[mask, "topic"], df.loc[mask, "kmeans"]))
            if mask.sum() >= 2 else None)
        diagnostics["bertopic"] = dict(bt_diagnostics, status="ok", embedding_cache=cache_info)
        topic_info.to_csv(TAB / "05_bertopic_themes.csv", index=False)
        display(topic_info)
        display(pd.Series(bt_diagnostics, name="Valeur").to_frame())
else:
    print("BERTopic non exécuté. Définir RUN_BERTOPIC = True après installation des dépendances optionnelles.")

## 3. Résultats et figures

Les fichiers d’affectation et de diagnostic sont réécrits pour cette exécution. Les identifiants des groupes ne sont pas alignés entre modèles ou entre exécutions. Les libellés sont des mots-clés automatiques, pas des résumés d’actualité validés.

In [ ]:
import matplotlib.pyplot as plt

df.to_csv(PROC / "post_topics.csv", index=False)
(PROC / "topic_diagnostics.json").write_text(json.dumps(diagnostics, ensure_ascii=False, indent=2), encoding="utf-8")
top = km_info.sort_values("posts").tail(14)
fig, ax = plt.subplots(figsize=(11, 7))
ax.barh(top["label"], top["posts"], color="#2a6fdb")
ax.set(title="Thèmes K-Means — taille des groupes", xlabel="Posts analysés")
fig.tight_layout()
fig.savefig(FIG / "05_kmeans_themes.png", dpi=140)
display(fig)
plt.close(fig)
print("Affectations : data/processed/post_topics.csv")
print("Diagnostics : data/processed/topic_diagnostics.json")